#Week 4 - In Class - Linear Model Selection and Regularization - Solved


In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn.linear_model as skl_lm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, LeaveOneOut, KFold, cross_val_score
from sklearn.metrics import mean_squared_error

from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, RidgeCV

import statsmodels.formula.api as smf
import statsmodels.api as sm

##College Data Set
For this example, we will use the College data set. We will predict the number of appliations received using the other variables in the set. We'll also use binary encoding for the categorical variable 'Private'.

In [3]:
url1 = 'https://raw.githubusercontent.com/dsahota-applied-data-analysis/data/main/College.csv'
df = pd.read_csv(url1).drop('Unnamed: 0', axis='columns')
df['Private'] = df['Private'].factorize(sort = True)[0]

In [4]:
df.head()

,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,1,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,1,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,1,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,1,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,1,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15


##Train/Test Split

First split the data into 90% for a training set and 10% for a test set.

In [9]:
X = df.drop(columns = ['Apps'])
y = df['Apps']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.1, random_state=0)
y.mean(), np.std(y, ddof = 1)

(3001.6383526383524, 3870.201484435291)

##Scaling

Since we are using regularization, we will rescale the predictors to have mean 0 and standard deviation 1. Note that we fit the scalar on the training data, but apply it to the training and test data.

In [6]:
scaler = StandardScaler().fit(X_train)
X_train_transformed = scaler.transform(X_train)
X_test_transformed = scaler.transform(X_test)

## Modeling

First, fit a linear model using least squares on the training set and report the test error obtained.

In [8]:
OLS = skl_lm.LinearRegression()
OLS.fit(X_train_transformed, y_train)
pred = OLS.predict(X_test_transformed)
rmse = math.sqrt(mean_squared_error(y_test, pred))
rmse

LinearRegression()

846.6620313952644

Next, fit a ridge regression model on the training set, with $\lambda$ chosen by cross-validation, and report the test error obtained

In [14]:
alphas = 10**np.linspace(10,-2,100)
kf = KFold(n_splits = 5)

ridgecv = skl_lm.RidgeCV(alphas=alphas, scoring='neg_mean_squared_error', cv = kf)
ridgecv.fit(X_train_transformed, y_train)

# This is our optimal alpha which we will use to fit on all of the training data and make predictions.
ridgecv.alpha_


optimal_ridge = skl_lm.Ridge()
optimal_ridge.set_params(alpha=ridgecv.alpha_)
optimal_ridge.fit(X_train_transformed, y_train)
pred = optimal_ridge.predict(X_test_transformed)
rmse = math.sqrt(mean_squared_error(y_test, pred))
rmse

RidgeCV(alphas=array([1.00000000e+10, 7.56463328e+09, 5.72236766e+09, 4.32876128e+09,
       3.27454916e+09, 2.47707636e+09, 1.87381742e+09, 1.41747416e+09,
       1.07226722e+09, 8.11130831e+08, 6.13590727e+08, 4.64158883e+08,
       3.51119173e+08, 2.65608778e+08, 2.00923300e+08, 1.51991108e+08,
       1.14975700e+08, 8.69749003e+07, 6.57933225e+07, 4.97702356e+07,
       3.76493581e+07, 2.84803587e+0...
       2.00923300e+00, 1.51991108e+00, 1.14975700e+00, 8.69749003e-01,
       6.57933225e-01, 4.97702356e-01, 3.76493581e-01, 2.84803587e-01,
       2.15443469e-01, 1.62975083e-01, 1.23284674e-01, 9.32603347e-02,
       7.05480231e-02, 5.33669923e-02, 4.03701726e-02, 3.05385551e-02,
       2.31012970e-02, 1.74752840e-02, 1.32194115e-02, 1.00000000e-02]),
        cv=KFold(n_splits=5, random_state=None, shuffle=False),
        scoring='neg_mean_squared_error')

0.01

Ridge(alpha=0.01)

Ridge(alpha=0.01)

846.582672979757

Now let's do the same for lasso, and report how many coefficients are non-zero

In [15]:
lassocv = skl_lm.LassoCV(alphas=alphas, cv = kf)
lassocv.fit(X_train_transformed, y_train)

# This is our optimal alpha which we will use to fit on all of the training data and make predictions.
lassocv.alpha_


optimal_lasso = skl_lm.Lasso()
optimal_lasso.set_params(alpha=lassocv.alpha_)
optimal_lasso.fit(X_train_transformed, y_train)
pred = optimal_lasso.predict(X_test_transformed)
rmse = math.sqrt(mean_squared_error(y_test, pred))
optimal_lasso.coef_
(optimal_lasso.coef_!=0).sum()
rmse

LassoCV(alphas=array([1.00000000e+10, 7.56463328e+09, 5.72236766e+09, 4.32876128e+09,
       3.27454916e+09, 2.47707636e+09, 1.87381742e+09, 1.41747416e+09,
       1.07226722e+09, 8.11130831e+08, 6.13590727e+08, 4.64158883e+08,
       3.51119173e+08, 2.65608778e+08, 2.00923300e+08, 1.51991108e+08,
       1.14975700e+08, 8.69749003e+07, 6.57933225e+07, 4.97702356e+07,
       3.76493581e+07, 2.84803587e+0...
       2.00923300e+00, 1.51991108e+00, 1.14975700e+00, 8.69749003e-01,
       6.57933225e-01, 4.97702356e-01, 3.76493581e-01, 2.84803587e-01,
       2.15443469e-01, 1.62975083e-01, 1.23284674e-01, 9.32603347e-02,
       7.05480231e-02, 5.33669923e-02, 4.03701726e-02, 3.05385551e-02,
       2.31012970e-02, 1.74752840e-02, 1.32194115e-02, 1.00000000e-02]),
        cv=KFold(n_splits=5, random_state=None, shuffle=False))

32.745491628777316

Lasso(alpha=32.745491628777316)

Lasso(alpha=32.745491628777316)

array([-1.77004976e+02,  3.54390710e+03, -5.18670862e+01,  5.20251229e+02,
       -0.00000000e+00, -0.00000000e+00,  3.16097573e+00, -1.48044889e+02,
        1.26856024e+02,  0.00000000e+00,  0.00000000e+00, -5.86220666e+01,
       -6.11174656e+01,  0.00000000e+00, -1.53164652e+01,  3.06917072e+02,
        5.97054227e+01])

12

807.5604365981859

Now, perform a PCR on the training set, with $M$ chosen by cross-validation, and report the test error along with the chosen value of $M$.

In [16]:
pca = PCA()
X_train_reduced = pca.fit_transform(X_train_transformed)

regr = skl_lm.LinearRegression()

rmse = []

for i in np.arange(1, X_train_reduced.shape[1]):
    score = math.sqrt(-1*cross_val_score(regr, X_train_reduced[:,:i], y_train, cv=kf, scoring='neg_mean_squared_error').mean())
    rmse.append(score)

rmse_per_component = pd.Series(np.array(rmse).flatten(), index = np.arange(1, X_train_reduced.shape[1]))
print(rmse_per_component)
np.amin(rmse_per_component)

1     3894.940401
2     2089.162263
3     2091.478871
4     1809.905571
5     1661.480145
6     1634.995828
7     1623.152653
8     1574.367592
9     1553.748973
10    1553.087763
11    1555.534459
12    1559.010798
13    1561.969337
14    1559.926111
15    1434.969998
16    1225.999437
dtype: float64


1225.9994367911047

In [17]:
# Since the optimal M is 16, we can continue.
M = 16
regr.fit(X_train_reduced[:, :M], y_train)
pred = regr.predict(pca.transform(X_test_transformed)[:, :M])
rmse = math.sqrt(mean_squared_error(y_test, pred))
rmse

LinearRegression()

819.1992303317376

Next, do the same for a PLS, with $M$ selected by cross validation, and report the test error and the chosen value of $M$.

In [18]:
rmse = []

for i in np.arange(1, X_train_transformed.shape[1]):
    score = math.sqrt(-1*cross_val_score(PLSRegression(i), X_train_transformed, y_train, cv=kf, scoring='neg_mean_squared_error').mean())
    rmse.append(score)

rmse_per_component = pd.Series(np.array(rmse).flatten(), index = np.arange(1, X_train_transformed.shape[1]))
print(rmse_per_component)
np.amin(rmse_per_component)

1     1913.055581
2     1633.787629
3     1488.277159
4     1422.798785
5     1299.076893
6     1224.725834
7     1212.382416
8     1201.425452
9     1196.127362
10    1197.941013
11    1198.961236
12    1198.298171
13    1198.733630
14    1198.763229
15    1198.673750
16    1198.552951
dtype: float64


1196.1273624704559

In [19]:
# Since the optimal M is 9, we can continue
M = 9
pls = PLSRegression(M)
pls.fit(X_train_transformed, y_train)
pred = pls.predict(X_test_transformed)
rmse = math.sqrt(mean_squared_error(y_test, pred))
rmse

PLSRegression(n_components=9)

831.9796953217609

What can you conclude? How accurately can we predict the number of applications? Is there much difference between the performance of these different models?